# Clustering Model Training


This notebook trains clustering models to identify demand patterns in daily electricity consumption profiles.

## Models Used:
- **StandardScaler**: Standardizes 96-point daily profile features
- **K-Means**: Groups similar daily demand profiles into clusters
- **DBSCAN**: Density-based clustering for pattern detection
- **Silhouette Score**: Evaluates clustering quality
- **Davies-Bouldin Index**: Another clustering quality metric

## Steps:
1. Load daily profiles from notebooks/outputs/daily_profiles.csv (made by notebook 02)
2. Select p00 to p95 features for clustering
3. Train StandardScaler and save to notebooks/outputs/models/profile_scaler.pkl
4. Test K-Means with K=2 to 12 and select best K (highest silhouette, same as clustering.py)
5. Train final K-Means and save to notebooks/outputs/models/kmeans_model.pkl
6. Train DBSCAN with eps from the k-distance knee (same as clustering.py) and save to notebooks/outputs/models/dbscan_model.pkl
7. Create clustering results and save metrics


> **Note (19 Oct 2026): this notebook saves its files to `notebooks/outputs/`, not `data/processed/` or `models/`.**
>
> The dashboard and API read `data/processed/` and `models/`. So that running this notebook can never overwrite those production files, every file it saves goes to `notebooks/outputs/` instead (models to `notebooks/outputs/models/`). Notebooks 01 → 05 read each other's files from that folder, so the chain still works end to end.
>
> The logic is the same as the app modules in `backend/app/ml/`. The last cell compares this notebook's results with the production files (read-only).

In [1]:

import pandas as pd
import numpy as np
import pickle
import json
from pathlib import Path
import sys

# Add project root to Python path
PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

# Import sklearn components
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score, davies_bouldin_score

# Set up paths
# Outputs go to notebooks/outputs/ so production data/processed/ and models/ are never overwritten
DATA_PROCESSED_DIR = PROJECT_ROOT / "notebooks" / "outputs"
MODELS_DIR = DATA_PROCESSED_DIR / "models"
DAILY_PROFILES_PATH = DATA_PROCESSED_DIR / "daily_profiles.csv"
CLUSTERING_RESULTS_PATH = DATA_PROCESSED_DIR / "clustering_results.csv"

# Create models directory
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Daily profiles path: {DAILY_PROFILES_PATH}")
print(f"Models directory: {MODELS_DIR}")


Project root: D:\Y4S1\Research\Component1-Behaviour-Intelligence\behaviour-intelligence
Daily profiles path: D:\Y4S1\Research\Component1-Behaviour-Intelligence\behaviour-intelligence\notebooks\outputs\daily_profiles.csv
Models directory: D:\Y4S1\Research\Component1-Behaviour-Intelligence\behaviour-intelligence\notebooks\outputs\models


In [2]:

# Load daily profiles
if not DAILY_PROFILES_PATH.exists():
    raise FileNotFoundError(f"Daily profiles not found at {DAILY_PROFILES_PATH}")

daily_profiles = pd.read_csv(DAILY_PROFILES_PATH)

print(f"Daily profiles loaded successfully!")
print(f"Shape: {daily_profiles.shape}")
print(f"Date range: {daily_profiles['date'].min()} to {daily_profiles['date'].max()}")

# Get profile columns
profile_cols = [f"p{str(i).zfill(2)}" for i in range(96)]
available_cols = [col for col in profile_cols if col in daily_profiles.columns]

print(f"Profile columns available: {len(available_cols)}")
print(f"\nFirst few rows:")
print(daily_profiles[['date', 'daily_mean', 'daily_peak']].head())


Daily profiles loaded successfully!
Shape: (1977, 106)
Date range: 2020-01-01 to 2025-05-30
Profile columns available: 96

First few rows:
         date   daily_mean  daily_peak
0  2020-01-01  1712.732396     2518.58
1  2020-01-02  1712.600938     2552.94
2  2020-01-03  1713.154375     2531.82
3  2020-01-04  1412.571979     2121.80
4  2020-01-05  1401.648021     1993.77


In [3]:

# Prepare features for clustering
X = daily_profiles[available_cols].values

print(f"Features prepared for clustering:")
print(f"- Number of samples (days): {X.shape[0]}")
print(f"- Number of features (time slots): {X.shape[1]}")
print(f"- Feature shape: {X.shape}")

# Check for missing values
missing_values = np.isnan(X).sum()
print(f"- Missing values: {missing_values}")

if missing_values > 0:
    print("Warning: Missing values found. Filling with mean values.")
    from sklearn.impute import SimpleImputer
    imputer = SimpleImputer(strategy='mean')
    X = imputer.fit_transform(X)
    print(f"Missing values after imputation: {np.isnan(X).sum()}")


Features prepared for clustering:
- Number of samples (days): 1977
- Number of features (time slots): 96
- Feature shape: (1977, 96)
- Missing values: 0


In [4]:

# Train StandardScaler
print("Training StandardScaler...")
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("StandardScaler trained successfully!")
print(f"Scaled data shape: {X_scaled.shape}")
print(f"Scaled data mean: {X_scaled.mean():.6f} (should be ~0)")
print(f"Scaled data std: {X_scaled.std():.6f} (should be ~1)")

# Save scaler
with open(MODELS_DIR / "profile_scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

print(f"Scaler saved to: {MODELS_DIR / 'profile_scaler.pkl'}")


Training StandardScaler...
StandardScaler trained successfully!
Scaled data shape: (1977, 96)
Scaled data mean: -0.000000 (should be ~0)
Scaled data std: 1.000000 (should be ~1)
Scaler saved to: D:\Y4S1\Research\Component1-Behaviour-Intelligence\behaviour-intelligence\notebooks\outputs\models\profile_scaler.pkl


In [5]:

# Find optimal K for K-Means
print("Testing K values from 2 to 12...")

k_range = range(2, 13)
silhouette_scores = []
davies_bouldin_scores = []

for k in k_range:
    print(f"Testing K={k}...", end=" ")
    
    # Fit K-Means
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=20)
    labels = kmeans.fit_predict(X_scaled)
    
    # Calculate metrics
    sil_score = silhouette_score(X_scaled, labels)
    db_score = davies_bouldin_score(X_scaled, labels)
    
    silhouette_scores.append(sil_score)
    davies_bouldin_scores.append(db_score)
    
    print(f"Silhouette: {sil_score:.4f}, DBI: {db_score:.4f}")

# Find best K based on silhouette score
best_k_idx = np.argmax(silhouette_scores)
best_k = k_range[best_k_idx]
best_silhouette = silhouette_scores[best_k_idx]
best_dbi = davies_bouldin_scores[best_k_idx]

print(f"\nBest K selected: {best_k}")
print(f"Silhouette Score: {best_silhouette:.4f}")
print(f"Davies-Bouldin Index: {best_dbi:.4f}")


Testing K values from 2 to 12...
Testing K=2... 

Silhouette: 0.4729, DBI: 0.8373
Testing K=3... Silhouette: 0.3839, DBI: 1.0281
Testing K=4... 

Silhouette: 0.2770, DBI: 1.3785
Testing K=5... Silhouette: 0.3095, DBI: 1.7812
Testing K=6... 

Silhouette: 0.3371, DBI: 2.1552
Testing K=7... Silhouette: 0.1705, DBI: 2.3790
Testing K=8... 

Silhouette: 0.2520, DBI: 2.0107
Testing K=9... Silhouette: 0.1933, DBI: 2.0667
Testing K=10... 

Silhouette: 0.2537, DBI: 1.8219
Testing K=11... Silhouette: 0.2234, DBI: 1.8350
Testing K=12... 

Silhouette: 0.1614, DBI: 1.9742

Best K selected: 2
Silhouette Score: 0.4729
Davies-Bouldin Index: 0.8373


In [6]:

# Train final K-Means model
print(f"Training final K-Means model with K={best_k}...")

kmeans_final = KMeans(n_clusters=best_k, random_state=42, n_init=20)
kmeans_labels = kmeans_final.fit_predict(X_scaled)

# Analyze cluster distribution
unique_labels, counts = np.unique(kmeans_labels, return_counts=True)
cluster_distribution = dict(zip(unique_labels, counts))

print(f"K-Means model trained successfully!")
print(f"\nCluster distribution:")
for cluster_id, count in sorted(cluster_distribution.items()):
    percentage = count / len(kmeans_labels) * 100
    print(f"Cluster {cluster_id}: {count} days ({percentage:.1f}%)")

# Save K-Means model
with open(MODELS_DIR / "kmeans_model.pkl", "wb") as f:
    pickle.dump(kmeans_final, f)

print(f"K-Means model saved to: {MODELS_DIR / 'kmeans_model.pkl'}")


Training final K-Means model with K=2...
K-Means model trained successfully!

Cluster distribution:
Cluster 0: 1396 days (70.6%)
Cluster 1: 581 days (29.4%)
K-Means model saved to: D:\Y4S1\Research\Component1-Behaviour-Intelligence\behaviour-intelligence\notebooks\outputs\models\kmeans_model.pkl


In [7]:
# Train DBSCAN model
# eps is set from the k-distance "knee" (same method as clustering.py _compute_knee_eps):
# take each day's distance to its 8th nearest neighbour and sort them; the knee is the
# point furthest from the straight line joining the first and last point of that curve.
print("Training DBSCAN model...")
from sklearn.neighbors import NearestNeighbors

min_samples = 8
nn = NearestNeighbors(n_neighbors=min_samples + 1)  # +1: each day is its own nearest neighbour
nn.fit(X_scaled)
distances, _ = nn.kneighbors(X_scaled)
k_dist = np.sort(distances[:, min_samples])

x = np.arange(len(k_dist))
x_n = (x - x.min()) / (x.max() - x.min())
y_n = (k_dist - k_dist.min()) / (k_dist.max() - k_dist.min())
num = np.abs((y_n[-1] - y_n[0]) * x_n - (x_n[-1] - x_n[0]) * y_n + x_n[-1] * y_n[0] - y_n[-1] * x_n[0])
den = np.sqrt((y_n[-1] - y_n[0]) ** 2 + (x_n[-1] - x_n[0]) ** 2)
knee_idx = int(np.argmax(num / den))
eps = float(k_dist[knee_idx])
print(f"k-distance knee at the {knee_idx / len(k_dist) * 100:.2f}th percentile -> eps = {eps:.4f}")

dbscan = DBSCAN(eps=eps, min_samples=min_samples)
dbscan_labels = dbscan.fit_predict(X_scaled)

# Analyze DBSCAN results
noise_count = np.sum(dbscan_labels == -1)
non_noise_labels = dbscan_labels[dbscan_labels != -1]
unique_dbscan_clusters = np.unique(non_noise_labels)

print(f"\nDBSCAN trained successfully!")
print(f"Final eps: {eps:.4f}")
print(f"Noise points: {noise_count} ({noise_count/len(dbscan_labels)*100:.1f}%)")
print(f"Number of clusters: {len(unique_dbscan_clusters)}")

# Save DBSCAN model
with open(MODELS_DIR / "dbscan_model.pkl", "wb") as f:
    pickle.dump(dbscan, f)

print(f"DBSCAN model saved to: {MODELS_DIR / 'dbscan_model.pkl'}")

Training DBSCAN model...
k-distance knee at the 96.81th percentile -> eps = 4.4183

DBSCAN trained successfully!
Final eps: 4.4183
Noise points: 60 (3.0%)
Number of clusters: 1
DBSCAN model saved to: D:\Y4S1\Research\Component1-Behaviour-Intelligence\behaviour-intelligence\notebooks\outputs\models\dbscan_model.pkl


In [8]:

# Create clustering results dataframe
clustering_results = pd.DataFrame({
    'date': daily_profiles['date'],
    'kmeans_cluster': kmeans_labels,
    'dbscan_cluster': dbscan_labels,
    'dbscan_noise_flag': (dbscan_labels == -1).astype(int),
    'daily_mean': daily_profiles['daily_mean'],
    'daily_peak': daily_profiles['daily_peak'],
    'daily_std': daily_profiles['daily_std'],
    'day_of_week': daily_profiles['day_of_week'],
    'month': daily_profiles['month'],
    'is_weekend': daily_profiles['is_weekend']
})

# Save clustering results
clustering_results.to_csv(CLUSTERING_RESULTS_PATH, index=False)

print(f"Clustering results saved to: {CLUSTERING_RESULTS_PATH}")
print(f"Results shape: {clustering_results.shape}")


Clustering results saved to: D:\Y4S1\Research\Component1-Behaviour-Intelligence\behaviour-intelligence\notebooks\outputs\clustering_results.csv
Results shape: (1977, 10)


In [9]:

# Calculate and save clustering metrics
clustering_metrics = {
    'best_k': int(best_k),
    'silhouette_score': round(float(best_silhouette), 4),
    'davies_bouldin_index': round(float(best_dbi), 4),
    'cluster_distribution': {int(k): int(v) for k, v in cluster_distribution.items()},
    'dbscan_noise_count': int(noise_count),
    'dbscan_cluster_count': int(len(unique_dbscan_clusters)),
    'dbscan_parameters': {
        'eps': round(float(eps), 4),
        'min_samples': min_samples
    },
    'data_info': {
        'total_days': len(daily_profiles),
        'features_used': len(available_cols),
        'date_range_start': daily_profiles['date'].min(),
        'date_range_end': daily_profiles['date'].max()
    }
}

# Save metrics
with open(MODELS_DIR / "clustering_metrics.json", 'w') as f:
    json.dump(clustering_metrics, f, indent=2)

print(f"Clustering metrics saved to: {MODELS_DIR / 'clustering_metrics.json'}")

print("\n=== CLUSTERING TRAINING SUMMARY ===")
print(f"Best K for K-Means: {best_k}")
print(f"Silhouette Score: {best_silhouette:.4f}")
print(f"Davies-Bouldin Index: {best_dbi:.4f}")
print(f"DBSCAN clusters: {len(unique_dbscan_clusters)}")
print(f"DBSCAN noise points: {noise_count}")
print(f"\nSaved files:")
print(f"- {MODELS_DIR / 'profile_scaler.pkl'}")
print(f"- {MODELS_DIR / 'kmeans_model.pkl'}")
print(f"- {MODELS_DIR / 'dbscan_model.pkl'}")
print(f"- {CLUSTERING_RESULTS_PATH}")
print(f"- {MODELS_DIR / 'clustering_metrics.json'}")


Clustering metrics saved to: D:\Y4S1\Research\Component1-Behaviour-Intelligence\behaviour-intelligence\notebooks\outputs\models\clustering_metrics.json

=== CLUSTERING TRAINING SUMMARY ===
Best K for K-Means: 2
Silhouette Score: 0.4729
Davies-Bouldin Index: 0.8373
DBSCAN clusters: 1
DBSCAN noise points: 60

Saved files:
- D:\Y4S1\Research\Component1-Behaviour-Intelligence\behaviour-intelligence\notebooks\outputs\models\profile_scaler.pkl
- D:\Y4S1\Research\Component1-Behaviour-Intelligence\behaviour-intelligence\notebooks\outputs\models\kmeans_model.pkl
- D:\Y4S1\Research\Component1-Behaviour-Intelligence\behaviour-intelligence\notebooks\outputs\models\dbscan_model.pkl
- D:\Y4S1\Research\Component1-Behaviour-Intelligence\behaviour-intelligence\notebooks\outputs\clustering_results.csv
- D:\Y4S1\Research\Component1-Behaviour-Intelligence\behaviour-intelligence\notebooks\outputs\models\clustering_metrics.json


In [10]:
# Check against the app's clustering results (read-only)
app_clusters = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "clustering_results.csv")

same_dates = list(app_clusters['date']) == list(clustering_results['date'])
same_kmeans = (app_clusters['kmeans_cluster'].values == kmeans_labels).all()
same_dbscan = (app_clusters['dbscan_cluster'].values == dbscan_labels).all()

print("=== CHECK AGAINST THE APP (data/processed/clustering_results.csv) ===")
print(f"Same dates: {same_dates}")
print(f"Same K-Means cluster for every day: {same_kmeans}  (sizes {dict(app_clusters['kmeans_cluster'].value_counts().sort_index())})")
print(f"Same DBSCAN label for every day: {same_dbscan}")

=== CHECK AGAINST THE APP (data/processed/clustering_results.csv) ===
Same dates: True
Same K-Means cluster for every day: True  (sizes {0: 1396, 1: 581})
Same DBSCAN label for every day: True
